## 1. End-to-End Pipeline Test
This notebook tests the full research pipeline: document upload → research query → quality verification.

In [ ]:
import os, sys, httpx, time
sys.path.insert(0, os.path.join(os.getcwd(), ".."))
from dotenv import load_dotenv
load_dotenv()

BACKEND_URL = os.getenv("API_URL", "http://localhost:8000")
print(f"🔗 Backend URL: {BACKEND_URL}")

## 2. Health Check
Verify the backend and MCP servers are running.

In [ ]:
response = httpx.get(f"{BACKEND_URL}/health", timeout=10)
health = response.json()
print(f"✅ Backend status: {health.get('status', 'unknown')}")
print(f"📋 MCP servers: {health.get('mcp_servers', {})}")

## 3. Upload a Document
Upload a sample PDF for the research pipeline.

In [ ]:
sample_doc = os.path.join(os.getcwd(), "..", "data", "sample_docs", "sample.pdf")
if not os.path.exists(sample_doc):
    print("⚠️ No sample document found. Skipping upload test.")
    upload_id = None
else:
    with open(sample_doc, "rb") as f:
        response = httpx.post(
            f"{BACKEND_URL}/upload",
            files={"files": ("sample.pdf", f, "application/pdf")},
            timeout=120,
        )
    result = response.json()
    upload_id = result.get("upload_id")
    print(f"✅ Upload started: {upload_id}")
    print(f"📄 Files: {result.get('files', [])}")

    # Poll for completion
    for _ in range(30):
        status = httpx.get(f"{BACKEND_URL}/upload/{upload_id}/status", timeout=10).json()
        if status.get("status") in ("completed", "error"):
            break
        time.sleep(2)
    print(f"📊 Upload status: {status.get('status')} — {status.get('message', '')}")

## 4. Run Research Query via SSE
Execute a research query and stream results.

In [ ]:
import json

query_payload = {
    "query": "Summarize the key concepts and methodology described in the uploaded document.",
    "quality_threshold": 6.0,
    "max_iterations": 2,
    "enable_web_search": False,
    "enable_sectioned": False,
}

print(f"🔬 Starting research: {query_payload['query'][:80]}...")
print("=" * 60)

events = []
with httpx.stream("POST", f"{BACKEND_URL}/research", json=query_payload, timeout=300) as response:
    for line in response.iter_lines():
        if line.startswith("data: "):
            try:
                event = json.loads(line[6:])
                events.append(event)
                evt_type = event.get("event", "")
                if evt_type == "step":
                    print(f"  {event.get('icon', '•')} {event.get('title', '')}")
                elif evt_type == "progress":
                    print(f"  📊 Iteration {event.get('iteration')}/{event.get('max_iterations')} | Score: {event.get('quality_score', 0):.1f}")
                elif evt_type == "complete":
                    print(f"\n✅ Research complete!")
                    print(f"⭐ Final score: {event.get('quality_score', 0):.1f}/10")
                    print(f"🔢 Iterations: {event.get('iterations', 0)}")
            except json.JSONDecodeError:
                pass

print(f"\n📊 Total SSE events received: {len(events)}")

## 5. Verify Research Quality
Check the final output meets minimum quality standards.

In [ ]:
content_events = [e for e in events if e.get("event") == "content"]
complete_events = [e for e in events if e.get("event") == "complete"]

if complete_events:
    final = complete_events[-1]
    score = final.get("quality_score", 0)
    threshold = query_payload["quality_threshold"]
    passed = score >= threshold

    print(f"📊 Quality Assessment:")
    print(f"  Score: {score:.1f}/10")
    print(f"  Threshold: {threshold}")
    print(f"  Result: {'✅ PASSED' if passed else '⚠️ BELOW THRESHOLD'}")
else:
    print("⚠️ No completion event received — check backend logs")

if content_events:
    final_content = content_events[-1].get("content", "")
    print(f"\n📄 Report length: {len(final_content):,} chars")
    print(f"📝 Preview (first 500 chars):\n{'─'*40}")
    print(final_content[:500])